# Load Silver Products and Orders

In [0]:
from pyspark.sql.functions import *

products_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/products/"

orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_products = spark.read.format("delta").load(products_silver_path)

df_orders = spark.read.format("delta").load(orders_silver_path)
display(df_products)
display(df_orders)

# Calculate product-level sales

In [0]:
product_sales = (
    df_orders
    .groupBy("product_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_sales"),
        avg("unit_price").alias("average_selling_price")
    )
)

display(product_sales)

# Add product information

In [0]:
df_product_sales = (
    product_sales
    .join(
        df_products.select(
            "product_id",
            "product_name",
            "brand",
            "category"
        ),
        on="product_id",
        how="left"
    )
)

In [0]:
df_product_sales = df_product_sales.select(
    "product_id",
    "product_name",
    "brand",
    "category",
    "total_orders",
    "total_quantity",
    "total_sales",
    "average_selling_price"
)
display(df_product_sales)

# Write to Gold

In [0]:
product_sales_gold_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/gold/product_sales/"
df_product_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(product_sales_gold_path)

#Verify

In [0]:
df_product_sales_gold = (
    spark.read
    .format("delta")
    .load(product_sales_gold_path)
)

display(df_product_sales_gold)
print("Product Sales records:", df_product_sales_gold.count())